In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names =  ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min'] # ignore short sessions (10, 24,25)

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-28 15:58:06,256|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-28 15:58:06,513|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-28 15:58:06,661|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-28 15:58:06,807|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-28 15:58:06,829|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-28 15:58:07,012|DEBUG|53778|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-28 15:58:07,012|DEBUG|53778|sessions_from_nas_parsing|sessionl

In [3]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
# fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-02-28 15:58:07,019|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-28 15:58:07,208|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-28 15:58:07,209|DEBUG|53778|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-28 15:58:07,215|INFO|53778|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-28 15:58:07,215|DEBUG|53778|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [4]:
behav.keys()

Index(['from_position_bin', 'to_position_bin', 'posbin_position',
       'posbin_velocity', 'posbin_acc', 'posbin_raw', 'posbin_yaw',
       'posbin_pitch', 'posbin_raw_500msMedian', 'posbin_yaw_500msMedian',
       'posbin_pitch_500msMedian', 'posbin_raw_abs_acc_500msMedian',
       'posbin_yaw_abs_acc_500msMedian', 'posbin_pitch_abs_acc_500msMedian',
       'posbin_RawYawPitch_abs_vel_sum',
       'posbin_RawYawPitch_abs_vel_sum_500msMedian',
       'posbin_RawYawPitch_abs_acc_sum_500msMedian',
       'posbin_YawPitch_abs_vel_sum_500msMedian',
       'posbin_YawPitch_abs_acc_sum_500msMedian', 'forward_vs_rotation_corr',
       'posbin_forward_prop', 'posbin_below_velocity_thr',
       'facecam_pose_nose_neck_body1_angle_velocity',
       'facecam_pose_body1_body2_body3_angle_velocity',
       'velocity_threshold_at_R1', 'velocity_threshold_at_R2',
       'facecam_pose_nose_x', 'facecam_pose_nose_y',
       'facecam_pose_nose_likelihood', 'facecam_pose_neck_x',
       'facecam_pose_ne

In [9]:
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events

2026-02-28 16:40:51,841|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-28 16:40:52,126|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-28 16:40:52,127|DEBUG|53778|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-28 16:40:52,139|INFO|53778|analytics|get_analytics
	Analytic `TrialWiseT0Events40ms` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-28 16:40:52,142|DEBUG|53778|analytics|get_analytics
	Processing TrialWiseT0Events40ms, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrack

trial_id  cue  trial_outcome  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              1.0  1.0            0.0   
                                       1              1.0  1.0            0.0   
                                       2              1.0  1.0            0.0   
                                       3              1.0  1.0            0.0   
                                       4              1.0  1.0            0.0   
...                                                   ...  ...            ...   
                      2025-01-27_13-39 1104         152.0  2.0            1.0   
                                       1105         152.0  2.0            1.0   
                                       1106         152.0  2.0            1.0   
                                       1107         152.0  2.0            1.0   
                                       1108         152.0  2.0            1.0   

                                                 choice_R1  choice_R2  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0               0.0        0.0   
                                       1               0.0        0.0   
                                       2               0.0        0.0   
                                       3               0.0        0.0   
                                       4               0.0        0.0   
...                                                    ...        ...   
                      2025-01-27_13-39 1104            0.0        1.0   
                                       1105            0.0        1.0   
                                       1106            0.0        1.0   
                                       1107            0.0        1.0   
                                       1108            0.0        1.0   

                                                     t0_event_name  \
paradigm_id animal_id session_id       entry_id                      
1100        6         2024-11-14_16-40 0           cueZone_visible   
                                       1             cueZone_entry   
                                       2              cueZone_exit   
                                       3         enter_reward1Zone   
                                       4         enter_reward2Zone   
...                                                            ...   
                      2025-01-27_13-39 1104           cueZone_exit   
                                       1105      enter_reward1Zone   
                                       1106      enter_reward2Zone   
                                       1107       exit_reward1Zone   
                                       1108       exit_reward2Zone   

                                                         t0  x_position  \
paradigm_id animal_id session_id       entry_id                           
1100        6         2024-11-14_16-40 0            4600000 -119.732067   
                                       1            5400000  -79.536247   
                                       2            6760000   24.898020   
                                       3            7080000   51.438920   
                                       4            9480000  170.376450   
...                                                     ...         ...   
                      2025-01-27_13-39 1104      4396000000   25.843260   
                                       1105      4396480000   49.770475   
                                       1106      4398880000  170.561500   
                                       1107      4397720000  109.711900   
                                       1108      4401200000  229.449700   

                                                 x_alignment  \
paradigm_id animal_id session_id       entry_id                
1100        6         2024-11-

In [6]:
behav['reward-valve-open_detected']

paradigm_id  animal_id  session_id        entry_id
1100         6          2024-11-14_15-01  0           0.0
                                          1           0.0
                                          2           0.0
                                          3           0.0
                                          4           0.0
                                                     ... 
                        2025-01-27_13-39  65051       0.0
                                          65052       0.0
                                          65053       0.0
                                          65054       0.0
                                          65055       0.0
Name: reward-valve-open_detected, Length: 1442177, dtype: float64

In [7]:
behavior = analytics.get_analytics(analytic="Behavior40msAligned", session_names=session_names)
behavior

2026-02-28 15:59:45,497|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-28 15:59:45,720|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-28 15:59:45,721|DEBUG|53778|analytics|get_analytics
	Processing Behavior40msAligned, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-28 15:59:45,725|INFO|53778|analytics|get_analytics
	Analytic `Behavior40msAligned` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-28 15:59:45,726|DEBUG|53778|analytics|get_analytics
	Processing Behavior40msAligned, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_2

frame_raw_500msMedian  \
paradigm_id animal_id session_id       entry_id                          
1100        6         2024-11-14_16-40 0                           NaN   
                                       1                           NaN   
                                       2                           NaN   
                                       3                           NaN   
                                       4                           NaN   
...                                                                ...   
                      2025-01-27_13-39 110898                      NaN   
                                       110899                      NaN   
                                       110900                      NaN   
                                       110901                      NaN   
                                       110902                      NaN   

                                                 frame_raw_abs_acc_500msMedian  \
paradigm_id animal_id session_id       entry_id                                  
1100        6         2024-11-14_16-40 0                                   NaN   
                                       1                                   NaN   
                                       2                                   NaN   
                                       3                                   NaN   
                                       4                                   NaN   
...                                                                        ...   
                      2025-01-27_13-39 110898                              NaN   
                                       110899                              NaN   
                                       110900                              NaN   
                                       110901                              NaN   
                                       110902                              NaN   

                                                 frame_YawPitch_abs_vel_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 0                                            NaN   
                                       1                                            NaN   
                                       2                                            NaN   
                                       3                                            NaN   
                                       4                                            NaN   
...                                                                                 ...   
                      2025-01-27_13-39 110898                                       NaN   
                                       110899                                       NaN   
                                       110900                                       NaN   
                                       110901                                       NaN   
                                       110902                                       NaN   

                                                 frame_YawPitch_abs_acc_sum_500msMedian  \
paradigm_id animal_id session_id       entry_id                                           
1100        6         2024-11-14_16-40 0                                            NaN   
                                       1                                            NaN   
                                       2                                            NaN   
                                       3                                            NaN   
                                       4                                            NaN   
...                                                                                 ...   
                      2025-01-27_13-39 110898                                       NaN   
                

In [ ]:
behavior.keys()

Index(['frame_raw_500msMedian', 'frame_raw_abs_acc_500msMedian',
       'frame_YawPitch_abs_vel_sum_500msMedian',
       'frame_YawPitch_abs_acc_sum_500msMedian',
       'frame_RawYawPitch_abs_vel_sum_500msMedian',
       'frame_RawYawPitch_abs_acc_sum_500msMedian', 'frame_forward_prop',
       'forward_vs_rotation_corr', 'frame_position',
       'facecam_pose_nose_neck_body1_angle_velocity',
       'facecam_pose_nose_neck_body1_angle',
       'facecam_pose_nose_neck_body1_angle_likelihood',
       'frame_ephys_timestamp', 'frame_pc_timestamp', 'trial_id', 'cue',
       'trial_outcome', 'choice_R1', 'choice_R2', 'lick_detected',
       'reward-sound_detected', 'reward-valve-open_detected', 'track_zone',
       'from_ephys_timestamp', 'to_ephys_timestamp'],
      dtype='object')

In [ ]:
reward_timings = behavior[behavior['reward-valve-open_detected'] == True]
reward_timings #['trial_id'].min()

1.0

In [ ]:
# display(t0_ens.head())
t0_events.head()

from_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0                      4400000   
                                       1                      4440000   
                                       2                      4480000   
                                       3                      4520000   
                                       4                      4560000   

                                                 to_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                       
1100        6         2024-11-14_16-40 0                    4440000   
                                       1                    4480000   
                                       2                    4520000   
                                       3                    4560000   
                                       4                    4600000   

                                                 Assembly001  Assembly002  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.044591    -0.079732   
                                       1            0.353522    -0.208656   
                                       2           -0.090020    -0.381483   
                                       3           -0.082682    -0.003392   
                                       4            0.169519    -0.213003   

                                                 Assembly003  Assembly004  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.117157    -0.100619   
                                       1           -0.050525    -0.098752   
                                       2            0.185447     1.103287   
                                       3            0.093962    -0.152654   
                                       4           -0.009496    -0.107002   

                                                 Assembly005  Assembly006  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.049611    -0.245620   
                                       1            0.562233     0.299666   
                                       2           -1.082371     1.418429   
                                       3           -0.208707     0.906473   
                                       4           -0.100514    -0.125251   

                                                 Assembly007  Assembly008  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            1.360649    -0.219548   
                                       1           -0.155457    -0.102385   
                                       2           -0.112464    -0.994913   
                                       3           -0.094875    -0.003057   
                                       4           -0.154239     0.000961   

                                                 ...  trial_id  cue  \
paradigm_id animal_id session_id       entry_id  ...                  
1100        6         2024-11-14_16-40 0         ...       1.0  1.0   
                                       1         ...       1.0  1.0   
                                       2         ...       1.0  1.0   
                                       3         ...       1.0  1.0   
                                       4         ...       1.0  1.0   

                                                 trial_outcome  choice_R1  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0                   0.0        0.0   
                                       1                   0.0        0.0   
                                       2                   0.0        0.0

trial_id  cue  trial_outcome  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              1.0  1.0            0.0   
                                       1              1.0  1.0            0.0   
                                       2              1.0  1.0            0.0   
                                       3              1.0  1.0            0.0   
                                       4              1.0  1.0            0.0   

                                                 choice_R1  choice_R2  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0               0.0        0.0   
                                       1               0.0        0.0   
                                       2               0.0        0.0   
                                       3               0.0        0.0   
                                       4               0.0        0.0   

                                                     t0_event_name       t0  \
paradigm_id animal_id session_id       entry_id                               
1100        6         2024-11-14_16-40 0           cueZone_visible  4600000   
                                       1             cueZone_entry  5400000   
                                       2              cueZone_exit  6760000   
                                       3         enter_reward1Zone  7080000   
                                       4         enter_reward2Zone  9480000   

                                                 x_position  x_alignment  \
paradigm_id animal_id session_id       entry_id                            
1100        6         2024-11-14_16-40 0        -119.732067       -120.0   
                                       1         -79.536247        -80.0   
                                       2          24.898020         25.0   
                                       3          51.438920         50.0   
                                       4         170.376450        170.0   

                                                       pre_cue_interval  ...  \
paradigm_id animal_id session_id       entry_id                          ...   
1100        6         2024-11-14_16-40 0         (4400000.0, 4600000.0]  ...   
                                       1                            NaN  ...   
                                       2                            NaN  ...   
                                       3                            NaN  ...   
                                       4                            NaN  ...   

                                                       R2_entry_interval  \
paradigm_id animal_id session_id       entry_id                            
1100        6         2024-11-14_16-40 0                             NaN   
                                       1                             NaN   
                                       2                             NaN   
                                       3                             NaN   
                                       4         (9080000.0, 10680000.0]   

                                                                            R2_entry_interval_bins  \
paradigm_id animal_id session_id       entry_id                                                      
1100        6         2024-11-14_16-40 0                                                      None   
                                       1                                                      None   
                                       2                                                      None   
                                       3                                                      None   
                                       4         [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1...   

                                                R1_exit_interval  \
parad

In [ ]:
# t0_ens['interval_name'].unique()
# print (t0_ens['interval_name'].unique())
# print(t0_events['t0_event_name'].unique())

['pre_cue_interval' 'nextto_cue_interval' 'cue_entry_interval'
 'cue_exit_interval' 'R1_entry_interval' 'R2_entry_interval'
 'R1_exit_interval' 'R2_exit_interval']
['cueZone_visible' 'cueZone_entry' 'cueZone_exit' 'enter_reward1Zone'
 'enter_reward2Zone' 'exit_reward1Zone' 'exit_reward2Zone' 'reward1_sound'
 'reward2_sound']


In [11]:
t0_ens = analytics.get_analytics('EnsembleT0Projection', session_names=session_names,)
t0_ens

2026-02-28 16:49:48,428|DEBUG|53778|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-28 16:49:48,750|DEBUG|53778|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-28 16:49:48,751|DEBUG|53778|analytics|get_analytics
	Processing EnsembleT0Projection, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-28 16:49:48,769|INFO|53778|analytics|get_analytics
	Analytic `EnsembleT0Projection` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-28 16:49:48,769|DEBUG|53778|analytics|get_analytics
	Processing EnsembleT0Projection, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackSto

from_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0                      4400000   
                                       1                      4440000   
                                       2                      4480000   
                                       3                      4520000   
                                       4                      4560000   
...                                                               ...   
                      2025-01-27_13-39 39795               4275440000   
                                       39796               4275480000   
                                       39797               4275520000   
                                       39798               4275560000   
                                       39799               4275600000   

                                                 to_ephys_timestamp  \
paradigm_id animal_id session_id       entry_id                       
1100        6         2024-11-14_16-40 0                    4440000   
                                       1                    4480000   
                                       2                    4520000   
                                       3                    4560000   
                                       4                    4600000   
...                                                             ...   
                      2025-01-27_13-39 39795             4275480000   
                                       39796             4275520000   
                                       39797             4275560000   
                                       39798             4275600000   
                                       39799             4275640000   

                                                 Assembly001  Assembly002  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.044591    -0.079732   
                                       1            0.353522    -0.208656   
                                       2           -0.090020    -0.381483   
                                       3           -0.082682    -0.003392   
                                       4            0.169519    -0.213003   
...                                                      ...          ...   
                      2025-01-27_13-39 39795       -0.121557    -0.456305   
                                       39796       -0.166517    -0.094205   
                                       39797       -0.062079    -0.142495   
                                       39798       -0.032098     0.095632   
                                       39799        0.065874     0.164919   

                                                 Assembly003  Assembly004  \
paradigm_id animal_id session_id       entry_id                             
1100        6         2024-11-14_16-40 0            0.117157    -0.100619   
                                       1           -0.050525    -0.098752   
                                       2            0.185447     1.103287   
                                       3            0.093962    -0.152654   
                                       4           -0.009496    -0.107002   
...                                                      ...          ...   
                      2025-01-27_13-39 39795        1.920625    -1.375872   
                                       39796        0.436531    -0.603980   
                                       39797       -0.051984    -0.121114   
                                       39798       -0.241508     0.670083   
                                       39799       -0.220040     0.440637   

                                                 Assembly005  Assembly006  \
paradigm_id animal_id session_id       entry_id                             
1100        6    

In [ ]:
t0_events['']
#t0_ens['interval_name'].unique()

array(['pre_cue_interval', 'nextto_cue_interval', 'cue_entry_interval',
       'cue_exit_interval', 'R1_entry_interval', 'R2_entry_interval',
       'R1_exit_interval', 'R2_exit_interval'], dtype=object)